<a href="https://colab.research.google.com/github/Janhvibabani/orbit-wars/blob/main/orbitwars.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade "kaggle-environments>=1.28.0"


In [ ]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

In [ ]:
# Run a quick game to see what the observation looks like
env = make("orbit_wars", debug=True)
env.run(["random", "random"])

# Peek at the initial observation
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

obs = env.steps[1][0].observation  # step 1 = first action step
planets = [Planet(*p) for p in obs.planets]
print(f"Player: {obs.player}")
print(f"Angular velocity: {obs.angular_velocity:.4f} rad/turn")
print(f"\nPlanets ({len(planets)}):")
for p in planets[:6]:
    owner_str = f"Player {p.owner}" if p.owner >= 0 else "Neutral"
    print(f"  id={p.id} owner={owner_str:10s} pos=({p.x:.1f}, {p.y:.1f}) r={p.radius:.1f} ships={p.ships} prod={p.production}")

In [ ]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def nearest_planet_sniper(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    # Separate our planets from targets
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        # Find the nearest planet we don't own
        nearest = None
        min_dist = float('inf')
        for t in targets:
            dist = math.sqrt((mine.x - t.x)**2 + (mine.y - t.y)**2)
            if dist < min_dist:
                min_dist = dist
                nearest = t

        if nearest is None:
            continue

        # How many ships do we need? Target's garrison + 1
        ships_needed = max(nearest.ships + 1, 20)

        # Only send if we have enough
        if mine.ships >= ships_needed:
            # Calculate angle from our planet to the target
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            moves.append([mine.id, angle, ships_needed])
    return moves

In [ ]:
env = make("orbit_wars", debug=True)
env.run([nearest_planet_sniper, "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

env.render(mode="ipython", width=800, height=600)


In [ ]:
env4 = make("orbit_wars", debug=True)
env4.run([nearest_planet_sniper, nearest_planet_sniper, nearest_planet_sniper, nearest_planet_sniper])

final = env4.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

env4.render(mode="ipython", width=800, height=600)

### **Helper functions**

In [ ]:
def dist(ax, ay, bx, by):
    return math.hypot(ax - bx, ay - by)


def fleet_speed(ships):
    if ships <= 1:
        return 1.0
    ratio = math.log(ships) / math.log(1000.0)
    ratio = max(0.0, min(1.0, ratio))
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)

### **Segment hits either sun or planet**

If you want to know where it hits sun use this pass the suns parameters and if you want it for planet pass the planets' parameters

In [ ]:
BOARD = 100.0
CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
SUN_SAFETY = 1.5
ROTATION_LIMIT = 50.0
TOTAL_STEPS = 500

def segment_hits_circle(x1, y1, x2, y2, cx, cy, radius, safety=0.0):
    r = radius + safety
    dx, dy = x2 - x1, y2 - y1
    fx, fy = x1 - cx, y1 - cy
    a = dx * dx + dy * dy


    if max(x1, x2) < cx - r:
        return False
    if min(x1, x2) > cx + r:
        return False
    if max(y1, y2) < cy - r:
        return False
    if min(y1, y2) > cy + r:
        return False

    if a < 1e-9:
        return math.hypot(x1 - cx, y1 - cy) < r

    b = 2 * (fx * dx + fy * dy)
    c = fx * fx + fy * fy - r * r

    disc = b * b - 4 * a * c
    if disc < 0:
        return False

    disc = math.sqrt(disc)
    t1 = (-b - disc) / (2 * a)
    t2 = (-b + disc) / (2 * a)

    return (0 <= t1 <= 1) or (0 <= t2 <= 1)

### **Fleets which are coming for our planets**

In [ ]:
def incoming_to_planet(planet, fleets):
    arrivals = []
    fvx_cache = {}
    for f in fleets:
        fvx, fvy = math.cos(f.angle), math.sin(f.angle)
        dx, dy = planet.x - f.x, planet.y - f.y
        proj = dx * fvx + dy * fvy
        if proj <= 0:
            continue
        perp = abs(dx * fvy - dy * fvx)
        if perp > planet.radius + 1.5:
            continue
        t = proj / fleet_speed(f.ships)
        if t > 80:
            continue
        arrivals.append((int(math.ceil(t)), f.owner, int(f.ships)))
    return arrivals

### **Predict mobile planets new position**

In [ ]:
def predict_planet_position(planet, initial_by_id, angular_velocity, turns):
    init = initial_by_id.get(planet.id)
    if init is None:
        return planet.x, planet.y
    orbital_r = dist(init.x, init.y, CENTER_X, CENTER_Y)
    if orbital_r + init.radius >= ROTATION_LIMIT:
        return planet.x, planet.y
    cur_ang = math.atan2(planet.y - CENTER_Y, planet.x - CENTER_X)
    new_ang = cur_ang + angular_velocity * turns
    return (CENTER_X + orbital_r * math.cos(new_ang),
            CENTER_Y + orbital_r * math.sin(new_ang))

### **Comet Helper functions**

In [ ]:
def comet_remaining_life(planet_id, comets):
  for group in comets:
    # finding the group which got planet_id
    pids = group.get("planet_ids", [])
    if planet_id not in pids:
      continue

    # get the idx of our planet_id
    idx = pids.index(planet_id)

    # All the paths -- we want paths of our planetid
    paths = group.get("paths", [])

    # Current position
    path_idx = group.get("path_index", 0)

    if idx < len(paths):
      return max(0, len(paths[idx]) - path_idx)

  return 0

In [ ]:
def predict_comet_position(planet_id, comets, turns):
  for group in comets:
    pids = group.get("planet_ids", [])
    if planet_id not in pids:
      continue

    idx = pids.index(planet_id)
    paths = group.get("paths", [])
    path_idx = group.get("path_index", 0)

    # Paths of our planet id
    path = paths[idx]

    # Bound check
    if idx >= len(paths):
      return None

    future_idx = path_idx + int(turns)
    if 0 <= future_idx < len(path):
      return path[future_idx][0], path[future_idx][1]
    return None
  return None

In [ ]:
def comet_gain(planet_id, comets, ships_needed, travel_time, remaining_life):

  if travel_time >= remaining_life:
    return -10**9

  usable_turns = remaining_life - travel_time
  future_ships = usable_turns * 1
  gain = future_ships - ships_needed

  return gain

In [ ]:
def evacuate_expiring_comets(my_planets, comet_ids, comets, initial_by_id, angular_velocity):
  moves = []
  evacuated_srcs = set()

  safe_planets = [p for p in my_planets if p.id not in comet_ids]
  if not safe_planets:
    return moves, evacuated_srcs

  for comet in my_planets:
    if comet.id not in comet_ids:
      continue
    if comet.ships <= 0:
      continue

    remaining_life = comet_remaining_life(comet.id, comets)
    if remaining_life > 3:
      continue

    dest = min(safe_planets, key=lambda p: dist(comet.x, comet.y, p.x, p.y))

    travel_turns = travel_time(comet.x, comet.y, dest.x, dest.y, comet.ships)
    if travel_turns >= remaining_life:
      continue

    fut_x, fut_y = predict_planet_position(dest, initial_by_id, angular_velocity, travel_turns)

    angle = math.atan2(fut_y - comet.y, fut_x - comet.x)

    if segment_hits_circle(comet.x, comet.y, fut_x, fut_y, 50.0, 50.0, 10.0, safety=0.5):
      continue

    moves.append([comet.id, float(angle), int(comet.ships)])
    evacuated_srcs.add(comet.id)

  return moves, evacuated_srcs

### **Agent**

In [ ]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

TOTAL_STEPS = 500

def agent(obs):
    moves = []

    # Extracting info
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    step = obs.get("step", 0) if isinstance(obs, dict) else obs.step

    planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    fleets = obs.get("fleets", []) if isinstance(obs, dict) else obs.fleets
    angular_velocity = obs.get("angular_velocity", 0) if isinstance(obs, dict) else obs.angular_velocity

    initial_planets = obs.get("initial_planets", []) if isinstance(obs, dict) else obs.initial_planets
    comets = obs.get("comets", []) if isinstance(obs, dict) else obs.comets
    comet_ids = obs.get("comet_planet_ids", []) if isinstance(obs, dict) else obs.comet_planet_ids

    # Converting into obj
    planets = [Planet(*p) for p in planets]
    fleets = [Fleet(*f) for f in fleets]

    planet_by_id = {p.id: p for p in planets}

    # Initial positions
    initial_by_id = {
        Planet(*p).id: Planet(*p)
        for p in initial_planets
    }

    # Splitting planets - mine vs target
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not my_planets:
        return []

    # Evacuate comets
    evac_moves, evac_srcs = evacuate_expiring_comets(
        my_planets,
        comet_ids,
        comets,
        initial_by_id,
        angular_velocity
    )
    moves.extend(evac_moves)

    # Defining game phases
    rem_steps = max(1, TOTAL_STEPS - step)

    early_phase = step < 60
    late_phase = rem_steps < 60
    mid_phase = not early_phase and not late_phase

    # Ships available for attack
    available = {}

    for mine in my_planets:
        total_ships = mine.ships
        incoming_fleets = incoming_to_planet(mine, fleets)
        defence = net_def_needed(mine, incoming_fleets, player)
        available[mine.id] = max(0, total_ships - defence)

    # Candidates
    candidates = []
    taken_targets = set()

    for mine in my_planets:
        if available[mine.id] <= 0:
            continue

        for target in targets:

            comet = target.id in comet_ids
            neutral = target.owner == -1
            enemy = target.owner != player and target.owner != -1

            distance = dist(mine.x, mine.y, target.x, target.y)

            travel_turns = travel_time(
                mine.x, mine.y,
                target.x, target.y,
                available[mine.id]
            )

            ships_needed = ships_needed_to_capture(target, travel_turns)
            profit_turns = rem_steps - travel_turns

            if profit_turns <= 0:
                continue
            if ships_needed > available[mine.id]:
                continue

            # Normal planet
            if not comet:
                future_prod = future_production(target, profit_turns)

                attack_score = future_prod * 100.0
                attack_score /= (ships_needed + 1.0)
                attack_score /= (travel_turns + 1.0)
                attack_score /= (1.0 + distance * 0.05)
                attack_score *= (profit_turns / TOTAL_STEPS)

            # Comets
            else:
                remaining_life = comet_remaining_life(target.id, comets)
                if travel_turns >= remaining_life:
                    continue

                usable_turns = remaining_life - travel_turns
                if usable_turns <= ships_needed:
                    continue

                gain = comet_gain(
                    target.id,
                    comets,
                    ships_needed,
                    travel_turns,
                    remaining_life
                )

                attack_score = gain * 100.0
                attack_score /= (1.0 + distance * 0.05)

            # ignore enemy comets
            if enemy and comet:
                continue

            # Phase adjustments
            if early_phase:
                if neutral and not comet:
                    attack_score *= 2.0
                elif neutral and comet:
                    attack_score *= 1.5
                elif enemy and not comet:
                    attack_score *= 0.6

            elif mid_phase:
                if enemy and not comet:
                    attack_score *= 1.2

            elif late_phase:
                if comet:
                    continue
                if enemy and not comet:
                    attack_score *= 2.5
                elif neutral and not comet:
                    attack_score *= 0.5

            candidates.append(
                (attack_score, mine.id, target.id, ships_needed, travel_turns)
            )

    # sort
    candidates.sort(reverse=True)

    # Launch attack
    for attack_score, source_id, target_id, ships_needed, travel_turns in candidates:

        if target_id in taken_targets:
            continue
        if available[source_id] < ships_needed:
            continue

        src = planet_by_id[source_id]
        target = planet_by_id[target_id]

        if target.id in comet_ids:
            future_x, future_y = predict_comet_position(
                target.id,
                comets,
                travel_turns
            )
        else:
            future_x, future_y = predict_planet_position(
                target,
                initial_by_id,
                angular_velocity,
                travel_turns
            )

        # Angle
        angle = math.atan2(future_y - src.y, future_x - src.x)

        # Sun check
        if segment_hits_circle(
            src.x, src.y,
            future_x, future_y,
            50.0, 50.0, 10.0,
            safety=0.5
        ):
            continue

        moves.append([
            source_id,
            float(angle),
            int(ships_needed)
        ])

        available[source_id] -= ships_needed
        taken_targets.add(target_id)

    return moves

### **Testing**

In [ ]:
# Test it against the random agent
env = make("orbit_wars", debug=True)
env.run([agent, "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

env.render(mode="ipython", width=800, height=600)

In [ ]:
Distance,time-travel,Production,defence ships needed to capture

In [ ]:

import math
from kaggle_environments.envs.orbit_wars.orbit_wars import (
    Planet,
Fleet)

# Board size
BOARD = 100.0

# Sun position (center of map)
CENTER_X = 50.0
CENTER_Y = 50.0

# Sun radius
SUN_R = 10.0

# Maximum fleet speed possible
MAX_SPEED = 6.0

# Extra safety distance around sun
SUN_SAFETY = 1.5

# Outer planets rotate less
ROTATION_LIMIT = 50.0

# Total game duration
TOTAL_STEPS = 500

def fleet_speed(
    num_ships: int,
    max_speed: float = MAX_SPEED
) -> float:


    # Prevent invalid or zero fleets
    if num_ships <= 1:
        return 1.0

    # Convert ship count into value between 0 and 1
    ratio = math.log(num_ships) / math.log(1000)

    # Curved speed scaling
    speed = 1.0 + (
        max_speed - 1.0
    ) * (ratio ** 1.5)

    return speed

def dist(
    x1: float,
    y1: float,
    x2: float,
    y2: float
) -> float:

    return math.hypot(
        x2 - x1,
        y2 - y1
    )


def travel_time(
    x1: float,
    y1: float,
    x2: float,
    y2: float,
    ships: int
) -> float:

    # Compute distance
    d = dist(
        x1,
        y1,
        x2,
        y2
    )

    # Prevent divide-by-zero
    if ships <= 0:
        return 999.0

    # Fleet speed depends on fleet size
    speed = fleet_speed(ships)

    # Final travel time
    return d / speed

def future_production(
    planet: Planet,
    turns: float
) -> int:

    return int(
        math.ceil(
            planet.production * turns
        )
    )

def ships_needed_to_capture(
    target: Planet,
    travel_turns: float,
    safety_margin: float = 1.05
) -> int:


    # Future defenders at arrival time
    future_garrison = (
        target.ships
        + future_production(
            target,
            travel_turns
        )
    )

    # Add safety buffer
    needed = int(
        math.ceil(
            future_garrison * safety_margin
        )
    ) + 1

    return max(1, needed)

def net_def_needed(
    planet: Planet,
    incoming_fleets: list,
    player_id: int,
    safety_margin: float = 1.15
) -> int:


    # Current defending ships
    garrison = planet.ships

    # Largest future deficit
    max_deficit = 0

    # Sort fleets by arrival time
    events = sorted(
        incoming_fleets,
        key=lambda x: x[0]
    )

    # Process all future arrivals
    for eta, owner, ships in events:

        # Planet produces ships until fleet arrives
        garrison += (
            eta * planet.production
        )

        # Enemy fleet attacks
        if owner != player_id:

            # Reduce defenders
            garrison -= ships

            # Planet would fall
            if garrison < 0:

                # Track largest deficit
                max_deficit = max(
                    max_deficit,
                    -garrison
                )

    # Add extra defensive safety
    reserve = int(
        math.ceil(
            max_deficit * safety_margin
        )
    )

    return max(0, reserve)